In [24]:
from dotenv import dotenv_values
from pathlib import Path

BATCH_ID = BASE_DIR = INPUT_DIR = OUTPUT_DIR = ""

config = dotenv_values(".env")
for key, value in config.items():
    if key.endswith("_DIR"):
        globals()[key] = Path(value)
    else:
        globals()[key] = value

In [25]:
from PIL import Image
import pandas as pd

def scan_images(directory):

    records = []

    for path in sorted(directory.iterdir()):

        if path.suffix.lower() not in [".jpg", ".jpeg"]:
            continue

        try:
            with Image.open(path) as img:

                records.append({
                    "filename": path.name,
                    "filepath": str(path),
                    "width": img.width,
                    "height": img.height,
                    "megapixels": round(
                        img.width * img.height / 1_000_000, 2
                    ),
                    "format": img.format,
                    "size_mb": round(
                        path.stat().st_size / 1_000_000, 2
                    )
                })

        except Exception as e:
            print(f"Error: {path.name}: {e}")

    return pd.DataFrame(records)


df_images = scan_images(INPUT_DIR)

display(df_images)
print("Total images:", len(df_images))

,filename,filepath,width,height,megapixels,format,size_mb
0,DSCF0438.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,14.19
1,DSCF0439.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,14.31
2,DSCF0447.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,12.75
3,DSCF0546.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,12.34
4,DSCF0556.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,12.42
5,DSCF0577.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,15.84
6,DSCF0578.JPG,/home/fahri/Projects/shutterstock_metadata/shu...,6000,4000,24.0,JPEG,15.80


Total images: 7


In [26]:
CATEGORY_GUIDELINES = """
OFFICIAL SHUTTERSTOCK CATEGORIES:

1. Abstract:
   Fractals, blur, patterns, textures and abstract concepts.

2. Animals/Wildlife:
   Domestic and wild animals.

3. Arts:
   Art objects, artwork, paintings and people making art.

4. Backgrounds/Textures:
   Images intended as backgrounds, textures, patterns,
   architectural details and flat lays.

5. Beauty/Fashion:
   Beauty products, hairstyles, clothing and fashion.

6. Buildings/Landmarks:
   Architectural structures, structural details
   and built environments.

7. Business/Finance:
   Businesspeople, workplaces, offices and financial subjects.

8. Celebrities:
   Celebrities and public figures.

9. Education:
   Schools, graduations and educational materials.

10. Food and Drink:
    Food, beverages, cooking and eating.

11. Healthcare/Medical:
    Health, wellness and medical subjects.

12. Holidays:
    Calendar events, celebrations, seasonal events and vacations.

13. Industrial:
    Construction, mining, electricity, tools,
    building materials and trades.

14. Interiors:
    Interior spaces of buildings and houses.

15. Miscellaneous:
    Subjects that do not fit another category.

16. Nature:
    Plants, animals, clouds, water and environmental subjects.

17. Objects:
    An object or group of objects as the main focus.

18. Parks/Outdoor:
    Parks, outdoor activities and camping.

19. People:
    Individuals, groups of people and body parts.

20. Religion:
    Religious and spiritual subjects.

21. Science:
    Scientific subjects, chemicals, living things and planets.

22. Signs/Symbols:
    Signs, flags, logos, icons and symbols.

23. Sports/Recreation:
    Sports, fitness and recreational hobbies.

24. Technology:
    Computers, phones, electronics and digital technology.

25. Transportation:
    Vehicles, roads, railways, aircraft and boats.

STRICT RULES:
- Select exactly 1 or 2 categories.
- Use ONLY the exact category names above.
- Never invent category names.
- Choose based on the actual visual subject.
- Do not select categories based on speculative context.
- Prefer the most specific relevant category.
"""

In [27]:
SHUTTERSTOCK_CATEGORIES = [
    "Abstract",
    "Animals/Wildlife",
    "Arts",
    "Backgrounds/Textures",
    "Beauty/Fashion",
    "Buildings/Landmarks",
    "Business/Finance",
    "Celebrities",
    "Education",
    "Food and Drink",
    "Healthcare/Medical",
    "Holidays",
    "Industrial",
    "Interiors",
    "Miscellaneous",
    "Nature",
    "Objects",
    "Parks/Outdoor",
    "People",
    "Religion",
    "Science",
    "Signs/Symbols",
    "Sports/Recreation",
    "Technology",
    "Transportation"
]

In [28]:
prompt = """
You are a professional stock photography
metadata specialist.

Analyze the provided photograph.

Generate Shutterstock metadata in JSON.

{SHUTTERSTOCK_CATEGORIES}

Required fields:
- Filename
- Description
- Keywords (separated by commas)
- Categories (1 or 2, separated by commas, must be selected from shutterstock categories list)
- Illustration (boolean)
- Mature Content (boolean)
- Editorial (boolean)
- review_required (boolean)
- review_reasons (why review_required are False)

Rules:
- Use valid understanable factual filename.
- Filename must be in English.
- Use factual descriptions.
- Description must be in English.
- Generate 15-30 relevant keywords.
- Minimum 7 and maximum 50 keywords.
- No duplicate keywords.
- Do not invent geographical information.
- Suggest valid categories.
- Categories only show label/name without category description.
- Flag visible logos and recognizable people.
- Do not make final legal eligibility decisions.
- Human approval is mandatory.

IMPORTANT:
Your task is to describe visual evidence,
NOT invent a story about the photograph.

STEP 1 - VISUAL IDENTIFICATION

Identify:
- Main physical object
- Secondary objects
- Visible people
- Environment
- Materials and textures
- Composition and depth of field

STEP 2 - CONTEXT CONTROL

Strict rules:
- Never infer occupation, identity, or legal status.
- Never assume a prison, hospital, factory, or other specific facility without visual evidence.
- Do not interpret people as prisoners, workers, customers, or criminals without evidence.
- Distinguish visible objects from inferred context.
- If uncertain, use a neutral physical description.

STEP 3 - STOCK METADATA

Generate:
- One factual description
- 20 relevant keywords
- Maximum 2 valid Shutterstock categories

{CATEGORY_GUIDELINES}

Do not use keywords based on speculation.

Return valid JSON only.
"""

In [29]:
METADATA_SCHEMA = {
    "type": "object",
    "properties": {
        "Filename": {"type": "string"},
        "Description": {"type": "string"},
        "Keywords": {
            "type": "array",
            "items": {"type": "string"}
        },
        "Categories": {
            "type": "array",
            "items": {"type": "string"}
        },
        "Illustration": {"type": "boolean"},
        "Mature Content": {"type": "boolean"},
        "Editorial": {"type": "boolean"},
        "review_required": {"type": "boolean"},
        "review_reasons": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": [
        "Filename",
        "Description",
        "Keywords",
        "Categories",
        "Illustration",
        "Mature Content",
        "Editorial",
        "review_required",
        "review_reasons"
    ]
}

In [30]:
import ollama
import base64
import json
from PIL import Image
from io import BytesIO

def analyze_image_resized(image_path: str, prompt: str, max_size: int) -> str:
    """Resize image before sending to reduce token count."""
    
    # Open and resize image
    with Image.open(image_path) as img:
        # Convert to RGB if needed
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Resize maintaining aspect ratio
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        
        # Save to bytes
        buffer = BytesIO()
        img.save(buffer, format='JPEG', quality=85)
        image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')
    
    # Send to Ollama
    response = ollama.chat(
        model='qwen2.5vl:3b',
        messages = [{
            'role': 'user',
            'content': prompt,
            'images': [image_base64]
        }],
        format=METADATA_SCHEMA,
        options = {
            "temperature": 0,
            "seed": 42,
            "num_predict": 512
        }
    )

    raw = response["message"]["content"]

    if not raw:
        raise ValueError("Empty model response")
    
    return json.loads(raw)

In [31]:
# import json

# for path in sorted(INPUT_DIR.iterdir()):
#     if path.suffix.lower() not in [".jpg", ".jpeg"]:
#         continue
#     # contents = [prompt, str(path)]
#     result = analyze_image_resized(path, prompt, max_size=512)

#     if result:
#         print(f"Successfully parsed metadata for {path.name}:")
#         print(json.dumps(result, indent=2))
#     else:
#         print(f"Failed to analyze {path.name}")

In [ ]:
import csv

records = []

for path in sorted(INPUT_DIR.iterdir()):
    if path.suffix.lower() not in [".jpg", ".jpeg"]:
        continue
    # contents = [prompt, str(path)]
    result = analyze_image_resized(path, prompt, max_size=512)

    if result:
        keywords_str = ", ".join(result.get("Keywords", []))
        categories_str = ", ".join(result.get("Categories", []))

        records.append({
            "Filename": result.get("Filename", path.name),
            "Description": result.get("Description", ""),
            "Keywords": keywords_str,
            "Categories": categories_str,
            "Editorial": result.get("Editorial", "")
        })
    else:
        print(f"Failed to analyze {path.name}")

csv_output_path = OUTPUT_DIR / "shutterstock_metadata.csv"

with open(csv_output_path, mode="w", encoding="utf-8", newline="") as f:
    if records:
        writer = csv.DictWriter(f, fieldnames=["Filename", "Description", "Keywords", "Categories", "Editorial"])
        writer.writeheader()
        writer.writerows(records)

print(f"CSV saved successfully to: {csv_output_path}")

df_result = pd.read_csv(csv_output_path)
display(df_result)